[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc2_donnees/cours/seance2_cours.ipynb)

# Séance 2.2 — Agréger, croiser et visualiser — étude de cas

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices et d'étude de cas)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- répondre à « combien par... ? » avec `groupby`, et calculer plusieurs indicateurs d'un coup
- rassembler plusieurs fichiers avec `merge`, sans perdre ni dupliquer de lignes
- choisir le bon graphique selon la question posée, et le rendre lisible
- repérer ce qu'un graphique cache autant que ce qu'il montre
- conclure une analyse par des recommandations chiffrées

## Retour à la question de départ

> *« Sur quel marché faut-il investir l'an prochain ? »*

Vous savez charger, comprendre et nettoyer. Et pourtant vous ne pouvez toujours
pas répondre — pour une raison très simple :

**`ventes.csv` ne contient pas le pays.** Il contient un `client_id`. Le pays
est dans `clients.csv`.

C'est la situation normale en entreprise : l'information est **répartie entre
plusieurs fichiers**, et la réponse naît de leur croisement. Cette séance mène
la question jusqu'au bout : croiser, agréger, puis **montrer** — et conclure.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc2_donnees/data/"

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")       ## une ligne = un produit
clients = pd.read_csv(BASE + "clients.csv")     ## une ligne = un client
produits = pd.read_csv(BASE + "produits.csv")   ## une ligne = une reference

ventes["ca"] = ventes["qte"] * ventes["prix"]     ## le CA de chaque ligne
ventes["date"] = pd.to_datetime(ventes["date"])   ## du texte vers des dates
print(ventes.shape, clients.shape, produits.shape)

## 1. Filtrer sur plusieurs conditions, puis trier

In [ ]:
grosses = ventes.query("qte >= 50 and prix < 2")   ## les deux a la fois
print(len(grosses), "lignes : beaucoup d'unites, prix unitaire faible")

# Attention aux guillemets : doubles a l'exterieur, simples a l'interieur
ue = clients.query("pays in ['France', 'Allemagne', 'Belgique']")   ## in
print(len(ue), "clients dans ces trois pays")

`and` empile deux conditions ; `in` teste l'appartenance à une liste et évite
d'écrire trois `or` à la suite.

In [ ]:
# ascending=False : du plus grand au plus petit
print(ventes.sort_values("ca", ascending=False).head(3)["ca"].tolist())

# nlargest fait la meme chose en plus court : trie + head, d'un coup
ventes.nlargest(3, "ca")[["prod_id", "qte", "ca"]]

## 2. `groupby` — la commande la plus utile de tout le bloc

`groupby` répond à toutes les questions de la forme **« combien par ... ? »**.

Trois temps, toujours les mêmes :

1. **Découper** les lignes en paquets selon une colonne
2. **Calculer** un indicateur dans chaque paquet
3. **Recoller** les résultats en un tableau

In [ ]:
# "Combien de chiffre d'affaires par client ?"
ca_client = ventes.groupby("client_id")["ca"].sum()   ## decouper, calculer, recoller

ca_client.nlargest(5).round(2)   ## les cinq plus gros clients

Un client pèse à lui seul **143 825 €**. Gardez ce chiffre en tête, on y
revient dans l'étude de cas.

### Plusieurs indicateurs d'un coup — `agg`

In [ ]:
resume = ventes.groupby("client_id").agg(
    ca=("ca", "sum"),              # total depense
    nb_lignes=("cmd_id", "count"), # nombre de LIGNES
    nb_cmd=("cmd_id", "nunique"),  # nombre de COMMANDES distinctes
)
resume.nlargest(3, "ca").round(2)

La syntaxe se lit : `nom_voulu=("colonne_source", "operation")`.

> ⚠️ **`count` vs `nunique` — l'erreur classique.**
> `count` compte les **lignes**. `nunique` compte les **valeurs distinctes**.
> Une commande de 30 articles occupe 30 lignes mais reste **une** commande.
> Regardez l'écart entre `nb_lignes` et `nb_cmd` ci-dessus : le premier client
> a 5 675 lignes pour 201 commandes. Confondre les deux, c'est diviser son
> panier moyen par 28.

Les opérations disponibles : `"sum"`, `"mean"`, `"median"`, `"min"`, `"max"`,
`"count"`, `"nunique"`, `"std"`.

## 3. `merge` — rassembler les fichiers

C'est l'équivalent du `RECHERCHEV` d'Excel, en beaucoup plus sûr.

Les deux tables ont une colonne en commun : `client_id`. `merge` s'en sert
pour aller chercher, pour chaque vente, les informations du client
correspondant.

In [ ]:
avant = len(ventes)
vc = ventes.merge(clients, on="client_id")   ## on = la colonne commune

# LE reflexe : verifier qu'on n'a ni perdu ni duplique de lignes
print(avant, "->", len(vc))
vc[["client_id", "pays", "segment", "qte", "prix", "ca"]].head(3)

> ⚠️ **Ne sautez jamais cette vérification.** Si la clé de jointure n'est pas
> unique dans la table de droite, `merge` **duplique** des lignes sans rien
> dire. Vos totaux deviennent faux et rien ne vous alerte. Deux nombres
> affichés, une seconde de lecture, et vous êtes tranquille.

### Et enfin, la réponse à la question du bloc

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

print(ca_pays.head(5).round(2))   ## le classement des marches
print("clients irlandais :", vc.query("pays == 'Irlande'")["client_id"].nunique())

Le Royaume-Uni domine — c'est le marché domestique, sans surprise.

**Mais regardez l'Irlande : 261 205 €, deuxième marché du groupe... pour
`2` clients.** Deux clients pèsent 22,7 % du chiffre d'affaires total.

Gardez cette anomalie de côté : c'est le point de départ de l'étude de cas de
cette séance. Une moyenne par pays ne veut rien dire quand deux clients font
le marché.

### Enchaîner les jointures

In [ ]:
# On ajoute maintenant les informations produit
complet = vc.merge(produits, on="prod_id")   ## deuxieme jointure, autre cle
print(len(complet), "lignes,", complet.shape[1], "colonnes")   ## toujours verifier

complet.groupby("categorie")["ca"].sum().nlargest(3).round(2)   ## le trio de tete

## 4. Croiser deux dimensions

`groupby` répond à « par pays ». Et « par pays **et** par segment » ?

In [ ]:
top4 = ca_pays.head(4).index   ## les etiquettes, pas les valeurs

vc.query("pays in @top4").pivot_table(
    values="ca",          ## ce qu'on calcule
    index="pays",         ## ce qui va en lignes
    columns="segment",    ## ce qui va en colonnes
    aggfunc="sum",        ## comment on l'agrege
).round(0)

> 💡 Le `@top4` dans `query()` veut dire « va chercher la variable Python
> nommée `top4` ». Pratique pour ne pas retaper une liste.

Deux cases sont vides pour l'Irlande. Ce n'est pas un bug : ses deux clients
sont tous deux classés « premium », il n'y a donc **aucune** ligne
« occasionnel » ou « standard » à additionner. **Une case vide dans un tableau
croisé est une information**, pas une erreur.

## 5. Pourquoi faire un graphique

Voici le chiffre d'affaires mensuel, sous forme de tableau :

| mois | CA | mois | CA |
|---|---|---|---|
| 2010-12 | 57 705 | 2011-06 | 75 590 |
| 2011-01 | 78 452 | 2011-07 | 107 164 |
| 2011-02 | 52 115 | 2011-08 | 85 469 |
| 2011-03 | 80 451 | 2011-09 | 133 236 |
| 2011-04 | 60 493 | 2011-10 | 170 010 |
| 2011-05 | 80 393 | 2011-11 | 133 937 |

Vous l'avez lu. Avez-vous **vu** quelque chose ?

Regardez maintenant la même chose en courbe, dans la cellule suivante. La
tendance saute aux yeux en une seconde.

> **Un graphique ne décore pas un rapport : il fait voir ce qu'un tableau
> cache.** Corollaire souvent oublié — si un tableau de trois lignes suffit,
> ne faites pas de graphique.

### Choisir le bon graphique

| Votre question | Le graphique |
|---|---|
| Comment ça évolue **dans le temps** ? | une **courbe** |
| Qui est le plus gros ? Comment ça se **compare** ? | des **barres** |
| Comment les valeurs sont-elles **réparties** ? | un **histogramme** |
| Y a-t-il un **lien** entre deux grandeurs ? | un **nuage de points** |

Quatre questions, quatre graphiques. C'est presque tout ce dont vous aurez
besoin.

## 6. La courbe — l'évolution dans le temps

In [ ]:
# to_period("M") regroupe toutes les dates d'un meme mois
ca_mois = ventes.groupby(ventes["date"].dt.to_period("M"))["ca"].sum()
ca_mois.index = ca_mois.index.astype(str)   ## en texte : matplotlib prefere

ca_mois.plot(kind="line", marker="o", figsize=(7, 4))   ## une evolution
plt.title("Chiffre d'affaires mensuel")   ## sans titre, ce n'est pas un livrable
plt.ylabel("CA (euros)")                  ## la grandeur ET son unite
plt.xticks(rotation=45)                   ## des dates inclinees se lisent
plt.tight_layout()
plt.show()

Une montée régulière jusqu'à un pic en octobre, puis une chute brutale en
décembre.

**Que concluez-vous ?** Prenez trente secondes avant de continuer.

In [ ]:
# Verifions quelque chose avant de conclure
decembre = ventes.query("date >= '2011-12-01'")

print("derniere date du fichier :", ventes["date"].max().date())   ## le 9 !
print("jours de decembre 2011 presents :", decembre["date"].dt.day.nunique())

> ⚠️ **Il n'y a pas eu d'effondrement en décembre.** Le fichier s'arrête au
> **9 décembre**. On compare 8 jours de vente à des mois complets de 30 jours.
>
> Le graphique ne mentait pas. C'est la lecture qui était fausse.

C'est **l'erreur d'analyse la plus fréquente en entreprise**, et l'une des
plus coûteuses. Avant d'interpréter une évolution, vérifiez toujours que
**toutes les périodes sont comparables**.

Le vrai pic, lui, est bien réel : octobre. Pour un grossiste, c'est logique —
les détaillants se réapprovisionnent **avant** Noël, pas pendant.

## 7. Les barres — comparer

**Toujours trier avant de tracer.** Un diagramme en barres non trié est
illisible.

In [ ]:
# nlargest(8) : le top 8. sort_values() : matplotlib dessine de bas en haut
ca_pays8 = complet.groupby("pays")["ca"].sum().nlargest(8).sort_values()

ca_pays8.plot(kind="barh", figsize=(7, 4), color="#4C72B0")
plt.title("Chiffre d'affaires par pays, 2011 (top 8)")   ## quoi + quand
plt.xlabel("Chiffre d'affaires (euros)")   ## la grandeur ET son unite
plt.ylabel("")            ## "pays" n'apprend rien : on l'enleve
plt.tight_layout()        ## rien ne sera coupe au bord de la figure
plt.show()

Quatre détails séparent un brouillon d'un livrable, et ils sont tous dans la
cellule ci-dessus :

- **`title`** : ce que montre le graphique, **et sur quelle période**
- **`xlabel`** : le nom de la grandeur **et son unité**
- **`ylabel("")`** : on enlève l'étiquette évidente
- **`tight_layout()`** : évite que les étiquettes soient coupées

> 💡 **`barh` plutôt que `bar`.** En barres horizontales, les noms se lisent
> sans se chevaucher et sans rotation. Et `sort_values()` **sans**
> `ascending=False` : matplotlib dessine de bas en haut, donc trier en ordre
> croissant met le plus grand tout en haut.

Si votre lecteur doit vous demander « c'est en quoi ? » ou « sur quelle
période ? », le graphique a échoué.

## 8. L'histogramme — la répartition

In [ ]:
ventes["prix"].plot(kind="hist", bins=50, figsize=(7, 4))   ## 50 classes
plt.title("Repartition des prix unitaires")
plt.xlabel("Prix (euros)")
plt.show()

Illisible : une seule barre collée à gauche. En cause, le prix maximum à
4 161 €, qui étire tout l'axe.

**C'est une information, pas un problème.** Elle confirme ce qu'on avait vu en
séance 2.1 (moyenne 3,93 € contre médiane 1,95 €). Zoomons sur la zone utile :

In [ ]:
# 85,8 % des ventes sont a moins de 5 euros
ventes.query("prix < 10")["prix"].plot(kind="hist", bins=40, figsize=(7, 4))

# Le zoom se dit DANS le titre : sinon on cache une information au lecteur
plt.title("Repartition des prix unitaires (moins de 10 euros)")
plt.xlabel("Prix (euros)")
plt.show()

Voilà l'entreprise réelle : **un vendeur de petits articles à moins de 5 €**,
en gros volumes. Ce n'est pas ce qu'une moyenne de 3,93 € laissait deviner —
elle aurait pu décrire aussi bien un catalogue homogène autour de 4 €.

> Quand vous zoomez pour rendre un graphique lisible, **dites-le dans le
> titre**. Sans la mention « moins de 10 euros », vous cachez une information
> à votre lecteur.

## 9. Le nuage de points — chercher un lien

In [ ]:
# random_state=42 : le meme echantillon a chaque execution
echantillon = ventes.query("prix < 20 and qte < 200").sample(2000, random_state=42)

# alpha=0.3 : des points translucides, sinon 2 000 points font une tache
echantillon.plot(kind="scatter", x="prix", y="qte", alpha=0.3, figsize=(7, 4))
plt.title("Quantite commandee selon le prix unitaire (zone courante)")
plt.xlabel("Prix unitaire (euros)")
plt.ylabel("Quantite")
plt.show()

On croit voir que les grosses quantités se concentrent à gauche. Mesurons
plutôt que de croire :

In [ ]:
# corr() resume un nuage en un seul nombre, entre -1 et +1
print("sur la zone tracee   :", round(echantillon["prix"].corr(echantillon["qte"]), 3))
print("sur le fichier entier:", round(ventes["prix"].corr(ventes["qte"]), 3))

**−0,273 sur la zone tracée, −0,024 sur le fichier entier.** Le lien est
faible dans le premier cas et pratiquement inexistant dans le second.

Ce que cet écart enseigne vaut plus que le nuage lui-même : **c'est la fenêtre
choisie qui fabrique une partie de la relation.** En coupant à `prix < 20` et
`qte < 200`, on a retiré les articles chers commandés en grande quantité — et
la pente est apparue. Un graphique montre toujours ce qu'on a décidé de lui
donner.

> ⚠️ Et même si le lien avait été fort : **une relation n'est pas une cause.**
> Le prix ne « fait » pas baisser les quantités ; ce sont deux conséquences du
> type de produit.

---

## Et maintenant : l'étude de cas

Vous avez tous les outils. Passez au notebook d'exercices : les six
échauffements d'abord, l'étude de cas ensuite.

> 👥 **En binôme.** Un tient le clavier, l'autre lit l'énoncé et vérifie —
> puis vous échangez à mi-parcours. C'est la façon dont on travaille
> réellement sur une analyse, et c'est plus efficace que chacun de son côté.

---

## Ce que vous savez faire maintenant

### Agréger et croiser

| Vous voulez... | La commande |
|---|---|
| combiner des conditions | `df.query("prix > 10 and pays == 'France'")` |
| une liste de valeurs | `df.query("pays in ['France', 'Belgique']")` |
| trier | `df.sort_values("ca", ascending=False)` |
| les 5 plus grands | `df.nlargest(5, "ca")` |
| total par groupe | `df.groupby("pays")["ca"].sum()` |
| plusieurs indicateurs | `df.groupby("pays").agg(ca=("ca", "sum"), n=("cmd_id", "nunique"))` |
| joindre deux tables | `a.merge(b, on="client_id")` |
| croiser deux dimensions | `df.pivot_table(values="ca", index="pays", columns="segment", aggfunc="sum")` |

### Visualiser

| Votre question | Le graphique | La commande |
|---|---|---|
| comment ça évolue ? | courbe | `serie.plot(kind="line")` |
| qui est le plus gros ? | barres | `serie.plot(kind="barh")` |
| comment c'est réparti ? | histogramme | `df["prix"].plot(kind="hist", bins=30)` |
| y a-t-il un lien ? | nuage de points | `df.plot(kind="scatter", x="qte", y="prix")` |

Et toujours :

```python
plt.title("Ce que montre le graphique")
plt.xlabel("Nom de l'axe (unite)")
plt.ylabel("Nom de l'axe (unite)")
plt.show()
```

## Les quatre erreurs à ne jamais commettre

1. **Faire un `merge` sans vérifier le nombre de lignes avant et après.**
   Un `merge` peut silencieusement dupliquer ou faire disparaître des lignes.
   `print(len(a), "->", len(fusion))` : une seconde, et vous dormez tranquille.

2. **Confondre `count` et `nunique`.** `count` compte les lignes,
   `nunique` compte les valeurs distinctes. Une commande de 30 articles,
   c'est 30 lignes mais **une** commande.

3. **Rendre un graphique sans titre ni unité.** Si votre lecteur doit vous
   demander « c'est en quoi ? », ce n'est pas un livrable.

4. **Interpréter une évolution sans vérifier que les périodes sont
   comparables.** C'est l'erreur d'analyse la plus fréquente en entreprise.

> **Un chiffre ne devient une recommandation que quand il est rattaché à une
> décision.** « L'Irlande fait 22,7 % du CA » est un constat. « 22,7 % du CA
> repose sur deux comptes, il faut sécuriser ces contrats » est une
> recommandation.